# 12.7 二分搜尋法手刻演算法一：猜數字模型與精確匹配

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/johnnyy-lab/APCS1to3/blob/main/PythAPCS123_12-7_binary_search_exact_match.ipynb)

**適合對象**：程式設計初學者（完全零基礎） / APCS 扎根學習者  
**先備知識**：已掌握 Chapter 12 前六節之排序機制、雙指標概念與線性搜尋 $O(N)$ 極限。

---

### 學習導覽：對數級的奇蹟——二分折半與猜數字演算法模型

在上一節（12.6 節）中，我們學習了地毯式的線性搜尋，並且推導出它的時間複雜度是 $O(N)$。
想像一下：如果今天全國戶政系統有 2,300 萬筆國民身分資料，如果每次查詢某個人都要從第 1 筆比對到第 2,300 萬筆，每秒鐘只能服務幾個人，全國電腦系統早就癱瘓了！

但如果這 2,300 萬筆資料**已經嚴格依照身分證字號由小到大排好序**呢？
這時，我們根本不需要傻傻地一筆一筆看！
大家在同樂會都玩過「猜數字遊戲（1 到 100 猜一個秘密數字）」：
聰明的你第一步絕對不會猜 1，而是大喊：「**50！**」
如果莊家說：「太大了！」
你瞬間就知道：秘密數字必定在 1 到 49 之間，後面 50 到 100 整整一半的數字**當場被永久淘汰**！
接著你猜 25，範圍又瞬間砍半變成 12 個數……
每一次詢問，候選範圍直接縮小一半！這就是名震天下的**二分搜尋法（Binary Search）**！

在本單元中，我們將透過 6 個平緩的微階梯，手把手推導並刻畫出精確二分搜尋演算法的完美鋼骨：
1. **12.7.1 二分搜尋前提鐵律**：資料必須已經嚴格排序！（未排序二分搜尋的慘劇）。
2. **12.7.2 猜數字遊戲物理模型**：雙指標 `left, right` 與折半搜尋區間。
3. **12.7.3 中間索引計算與防禦**：`mid = (left + right) // 2`。
4. **12.7.4 區間縮小與指針跳躍鐵律**：`left = mid + 1` 與 `right = mid - 1`，條件 `while left <= right:`。
5. **12.7.5 時間複雜度 $O(\log N)$ 的爆炸性威力**：為什麼 10 億筆資料只需 30 次比對？
6. **12.7.6 手刻精確二分搜尋完整實作**：標準模板與找不到時回傳 -1。

讓我們一起見證對數級演算法的極速魅力！

### 12.7.1 二分搜尋前提鐵律：資料必須已經嚴格排序！（未排序二分搜尋的慘劇）

#### 1. 生活故事比喻：翻閱字典 vs 翻閱打散的抽獎券
當你在翻查英文字典尋找 `"galaxy"` 時，你為什麼敢直接把字典從正中間啪一聲翻開？
因為你知道字典是嚴格依照 A 到 Z 排好的！如果你翻開看到 `"monkey"`，你敢 100% 肯定地把後半本整疊合上，眼睛只看前半本，因為在 `"monkey"` 後面絕對不可能出現 `"galaxy"`。
但想像一下：如果有人把一萬張抽獎彩券隨意攪亂塞在一個大摸彩箱裡，你隨手從摸彩箱中間抓出一張看見號碼是 500 號，你能因此把箱子左邊的所有彩券通通倒進垃圾桶、斷言裡面絕對沒有 300 號嗎？
**絕對不能！** 因為在混亂的世界中，300 號可能散落在箱子裡的任何一個角落！
這就是二分搜尋法的第一道鋼鐵戒律：**「先決條件——資料必須已經嚴格排序！」**

#### 2. 底層運作機制：單調性是「折半淘汰」的唯一依據
二分搜尋法之所以能在每一次比對中「果斷拋棄整整一半的資料」，唯一的數學邏輯依據是：
- 假設當前中間值為 $M = a[mid]$。
- 若數列保證升序，且我們的目標 $Target < M$：
  根據不等式的傳遞性，對於所有大於等於 $mid$ 的索引 $k$，必定滿足 $a[k] \ge M > Target$！
  因此，右半邊的所有元素 $a[mid \dots right]$ **全部保證不可能等於 Target**！演算法才能理直氣壯地直接將整個右半邊一刀切除！
如果數列沒有排序，這個不等式鏈條瞬間斷裂，一刀切下去，很可能直接把真正的解答當場丟進垃圾桶！

#### 3. 初學者常見陷阱：對未排序的串列直接執行二分搜尋
這是無數新手最常犯的重大盲點：
看到輸入了一串數字 `[45, 12, 89, 34, 67]`，沒有呼叫 `sort()`，就直接在上面套用二分搜尋迴圈。結果明明 `34` 在串列裡，程式卻輸出「找不到」！
請刻在腦海裡：**未排序的資料，只能使用上一節學到的線性搜尋 $O(N)$！想要二分搜尋，前置必須先排序！**

#### 4. APCS 實戰視野
APCS 觀念題中，常出現「以下何種情況可以使用二分搜尋法？」選項中若出現「未排序陣列」或「無規律鏈結串列」，通通都是誘答陷阱。牢記「排序是二分之母」，是演算法思維的根本分水嶺。

In [ ]:
# 範例 12.7.1：未排序數列二分搜尋引發的慘劇演示
# 一組「未排序」的數列，目標為 30（明明在索引 2！）
raw_nums = [50, 10, 30, 90, 70]
target = 30
print("未排序數列：", raw_nums)
print("我們想尋找的目標：", target)

# 模擬未排序時盲目二分搜尋：
mid = len(raw_nums) // 2  # 索引 2，數值為 30
# 雖然剛好中間是 30 找到了，但如果找 10 呢？
target2 = 10
# 中間值 raw_nums[2] 是 30，因為 10 < 30，二分邏輯會認為 10 在左半邊 [50, 10]
# 但如果原始數列是 [30, 90, 70, 10, 50]，中間是 70，左半是 [30, 90]，10 卻在右半邊！
# 二分搜尋會徹底把有 10 的那一半丟掉，導致「明明存在卻永遠找不到」的大悲劇！

# --- 正確的做法：嚴格先排序 ---
sorted_nums = sorted(raw_nums)
print("\n正道：先進行嚴格排序 ->", sorted_nums)
print("排序後具備單調性，二分搜尋保證 100% 正確命中！")

In [ ]:
# 填空題 12.7.1：確保二分搜尋前置條件成立
# 任務：在啟動二分搜尋前，務必對輸入資料完成升序排序。
user_inputs = [78, 12, 95, 34, 60]

# 請呼叫正確的方法對 user_inputs 進行原地升序排序
user_inputs.___()

print("具備單調性的已排序數列：", user_inputs)
print("首位最小值：", user_inputs[0], "，末位最大值：", user_inputs[-1])

In [ ]:
# ==========================================
# [4] Code 練習題 12.7.1
# 任務說明：
# 給定一個可能未排序的整數串列 data。
# 請撰寫程式：
# 1. 先檢驗 data 是否已經由小到大嚴格排序（非遞減）。
# 2. 若未排序，請印出 "檢驗未排序，正在執行自動排序修復..." 並將其排序。
# 3. 若已排序，印出 "已排序，可安全二分搜尋"。
# 4. 最後印出最終的 data。
#
# 【公開測試資料 1】
# data = [5, 2, 8, 1]
# 預期輸出：
# 檢驗未排序，正在執行自動排序修復...
# 最終串列： [1, 2, 5, 8]
#
# 【公開測試資料 2】
# data = [10, 20, 30]
# 預期輸出：
# 已排序，可安全二分搜尋
# 最終串列： [10, 20, 30]
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：
data = [5, 2, 8, 1]

is_sorted = True
for i in range(len(data) - 1):
    if data[i] > data[i + 1]:
        is_sorted = False
        break

if not is_sorted:
    print("檢驗未排序，正在執行自動排序修復...")
    data.sort()
else:
    print("已排序，可安全二分搜尋")

print("最終串列：", data)

In [ ]:
# ==========================================
# [5] Code 挑戰題 12.7.1
# 任務說明：
# 某題庫系統上傳了一批可能亂序的字串單字清單：
# words = ["orange", "apple", "banana", "watermelon", "grape"]
# 請寫出程式碼：
# 1. 將單字依字典序升序排序。
# 2. 印出排序後「最正中間位置（len // 2）」的單字，並說明若搜尋目標字典序小於該單字，下一步應往哪一半搜尋。
# （本題為自由挑戰題，無公開測資，請依題目情境自行思考並撰寫完整程式碼）
# ==========================================
# 請在下方撰寫你的程式碼：
words = ["orange", "apple", "banana", "watermelon", "grape"]
words.sort()
mid_idx = len(words) // 2
mid_word = words[mid_idx]

print("排序後單字清單：", words)
print(f"正中間單字（索引 {mid_idx}）為: '{mid_word}'")
print(f"若目標小於 '{mid_word}'，根據單調性，下一步應只在前半部（索引 0 ~ {mid_idx - 1}）搜尋！")

### 12.7.2 猜數字遊戲物理模型：雙指標 `left, right` 與搜尋區間折半

#### 1. 生活故事比喻：包圍圈的收縮戰術
想像警方在一條長度為 1,000 公尺的筆直封閉隧道中圍捕嫌犯。
為了徹底困死嫌犯，警方設置了兩堵防爆移動盾牌：
- 左盾牌（`left`）：立在隧道起點 0 公尺處。
- 右盾牌（`right`）：立在隧道終點 1,000 公尺處。
整個嫌犯可能藏身的「可疑區間」，就是這兩堵盾牌所包夾的閉區間 `[left, right]`！
現在指揮官派出一架無人機飛到兩堵盾牌正中間 `500 公尺（mid）` 處進行熱感應掃描：
- 感測回報：「嫌犯在無人機的東邊（比 500 更大的方向）！」
指揮官立刻下令：**「左盾牌前進！直接跳到 501 公尺處（`left = mid + 1`）！」**
這一瞬間，隧道前半段 0 到 500 公尺被永久封死排除，搜尋範圍立刻縮減了一半！

#### 2. 底層運作機制：閉區間 `[left, right]` 的生命週期
在二分搜尋法中，我們使用兩個整數變數定義當前搜尋的**有效閉區間 `[left, right]`**：
- **初始化**：
  `left = 0`（串列最開頭索引）。  
  `right = len(a) - 1`（串列最末尾索引）。  
  此時整個串列所有元素都在被包夾的視線範圍內。
- **動態收縮**：
  每一次計算中間點 `mid` 並比對後，`left` 或 `right` 會向中間大步跳躍，使整個區間長度 $right - left + 1$ 以幾何級數的速度驟降為原來的 $\frac{1}{2}$！
- **終止邊界**：
  當 `left > right` 時，代表兩堵盾牌在空間中發生了交叉錯位，可疑區間長度縮小為 0，證明全體資料已窮盡，目標確定不存在！

#### 3. 初學者常見陷阱：初始化把 `right` 寫成 `len(a)`
初學者最容易犯的邊界錯誤：
```python
right = len(a)  # 致命！超出索引上限！
```
如果定義的是閉區間，最後一個合法元素的位置是 `len(a) - 1`！如果寫成 `len(a)`，後續一旦計算到 `mid == len(a)`，就會直接引發 `IndexError` 崩潰。

#### 4. APCS 實戰視野
雙指標定義搜尋區間是所有高階二分演算法的靈魂架構。只要在腦海中清晰看見這兩堵盾牌的收縮過程，二分搜尋就再也不是冰冷的程式碼，而是一場勝券在握的包圍戰術！

In [ ]:
# 範例 12.7.2：觀察雙指標 left 與 right 的區間折半過程
# 已嚴格排序的數列
arr = [11, 23, 35, 47, 59, 71, 83, 95]
target = 71
print(f"已排序數列 (長度 {len(arr)}): {arr}")
print(f"尋找目標 target = {target}\n")

left = 0
right = len(arr) - 1
round_num = 1

while left <= right:
    mid = (left + right) // 2
    print(f"回合 {round_num}: 當前搜尋閉區間索引 [{left}, {right}]，涵蓋元素 {arr[left:right+1]}")
    print(f"  中間索引 mid = {mid}，數值 arr[{mid}] = {arr[mid]}")
    
    if arr[mid] == target:
        print(f"  🎯 賓果！在第 {round_num} 回合於索引 {mid} 命中目標！")
        break
    elif arr[mid] < target:
        print(f"  太小了（{arr[mid]} < {target}），目標在右半部！左盾牌推進至 mid + 1 = {mid + 1}")
        left = mid + 1
    else:
        print(f"  太大了（{arr[mid]} > {target}），目標在左半部！右盾牌退縮至 mid - 1 = {mid - 1}")
        right = mid - 1
    
    round_num += 1
    print("-" * 50)

In [ ]:
# 填空題 12.7.2：雙指標初始邊界設定
# 任務：為長度為 N 的已排序串列初始化二分搜尋的雙指標。
data = [3, 7, 14, 21, 35, 49]

# 左指標從 0 出發
left = ___
# 右指標必須指向最後一個有效元素（len(data) - 1）
right = len(data) - ___

print(f"初始化搜尋區間為索引 [{left}, {right}]")
print(f"對應首尾數值為: arr[{left}]={data[left]}, arr[{right}]={data[right]}")

In [ ]:
# ==========================================
# [4] Code 練習題 12.7.2
# 任務說明：
# 給定已排序數列 sorted_data 與目標 target。
# 請模擬印出「第一回合」的搜尋區間 [left, right]、中間索引 mid 與其數值，
# 並根據比對結果，印出下一回合 left 或 right 應該更新為多少。
#
# 【公開測試資料 1】
# sorted_data = [10, 20, 30, 40, 50, 60, 70]
# target = 20
# 預期輸出：
# 初回區間： [0, 6]
# 中間值： 40 (索引 3)
# 下一步更新： right = 2
#
# 【公開測試資料 2】
# sorted_data = [2, 4, 6, 8, 10]
# target = 9
# 預期輸出：
# 初回區間： [0, 4]
# 中間值： 6 (索引 2)
# 下一步更新： left = 3
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：
sorted_data = [10, 20, 30, 40, 50, 60, 70]
target = 20

l = 0
r = len(sorted_data) - 1
mid = (l + r) // 2

print(f"初回區間： [{l}, {r}]")
print(f"中間值： {sorted_data[mid]} (索引 {mid})")

if sorted_data[mid] > target:
    print(f"下一步更新： right = {mid - 1}")
elif sorted_data[mid] < target:
    print(f"下一步更新： left = {mid + 1}")
else:
    print("初回已直接命中！")

In [ ]:
# ==========================================
# [5] Code 挑戰題 12.7.2
# 任務說明：
# 請設計一個微型追蹤器：
# 給定一個包含 15 個整數的已排序串列：numbers = list(range(10, 160, 10))
# 搜尋一個不存在的目標 target = 75。
# 請使用 while 迴圈追蹤整個收縮過程，並記錄總共經歷了幾個回合才收斂結束（即 left > right）。
# 印出總經歷回合數。
# （本題為自由挑戰題，無公開測資，請依題目情境自行思考並撰寫完整程式碼）
# ==========================================
# 請在下方撰寫你的程式碼：
numbers = list(range(10, 160, 10))
target = 75

l, r = 0, len(numbers) - 1
steps = 0

while l <= r:
    steps += 1
    m = (l + r) // 2
    if numbers[m] == target:
        break
    elif numbers[m] < target:
        l = m + 1
    else:
        r = m - 1

print(f"搜尋目標 {target}（不存在），經歷了 {steps} 回合後確認無解（left > right 退出）！")

### 12.7.3 中間索引計算與防禦：`mid = (left + right) // 2`

#### 1. 生活故事比喻：切蛋糕的正中心下刀點
想像你手頭上有一根長度不等的長條年糕，兩位小朋友想要公平分享。
為了找出正中間的切割點，你會怎麼做？
你會量一下左端點刻度（`left`），再量一下右端點刻度（`right`），把兩個數字加起來除以 2：$\frac{left + right}{2}$，刀子就在那個正中間的刻度果斷切下去！
如果在整數世界裡除不盡呢？例如左端點在 2、右端點在 7：
$(2 + 7) / 2 = 4.5$。但在電腦陣列中，**索引必須是乾淨的整數**！不可能有 `a[4.5]` 這種東西！
這時，Python 的整數除法雙斜線 **`//`** 就發揮了神威：`9 // 2 == 4`，無條件自動向下取整，精準鎖定中間的整數索引！

#### 2. 底層運作機制：`// 2` 的無條件向下取整特性
在 Python 中，計算中間索引的標準語法為：
```python
mid = (left + right) // 2
```
讓我們觀察它的數學表現：
- **長度為奇數時**（如區間 `[0, 4]`，共 5 個元素）：  
  `mid = (0 + 4) // 2 = 2`。此時 `mid` 剛好落在最完美的正中央（左邊有 2 個、右邊有 2 個）！
- **長度為偶數時**（如區間 `[0, 5]`，共 6 個元素）：  
  `mid = (0 + 5) // 2 = 2`。此時偶數有兩個正中間元素（索引 2 與 3），`// 2` 會自動挑選偏左側的那一個作為代表，這在數學上完全合法且絲毫不影響二分折半的正確性！

#### 3. 初學者常見陷阱：忘記加小括號引發除法優先級悲劇
初學同學最常見的粗心低級失誤：
```python
mid = left + right // 2  # 致命錯誤！
```
因為在數學運算優先級中，除法 `//` 比加法 `+` 還要早執行！
這行代碼會被解讀為 `left + (right // 2)`！如果 `left = 4, right = 6`，本該是 $(4+6)//2 = 5$，卻算成了 $4 + 3 = 7$，直接跑到區間外面引發嚴重 Bug！請務必用小括號緊緊包覆分子：`(left + right) // 2`。

#### 4. APCS 實戰視野
在 C++ 或 Java 語言中，若數值極大可能發生整數溢位（Integer Overflow），需要寫成 `left + (right - left) // 2`。
但在 Python 3 中，整數具備「任意精度（Arbitrary Precision）」特性，永遠不會發生記憶體溢位！因此在 Python 中寫 `mid = (left + right) // 2` 既簡潔優雅又 100% 安全無虞。

In [ ]:
# 範例 12.7.3：中間索引計算與奇偶長度驗證
# 案例 A：長度為奇數（5 個元素，索引 0 到 4）
l1, r1 = 0, 4
mid1 = (l1 + r1) // 2
print(f"奇數區間 [{l1}, {r1}] 的正中間索引 mid = {mid1} (完美對稱)")

# 案例 B：長度為偶數（6 個元素，索引 0 到 5）
l2, r2 = 0, 5
mid2 = (l2 + r2) // 2
print(f"偶數區間 [{l2}, {r2}] 的中間索引 mid = {mid2} (偏左正中)")

# 示範小括號忘記加的大災難
wrong_mid = l2 + r2 // 2
print(f"💥 忘記小括號的算式 left + right // 2 結果為: {wrong_mid} (嚴重出界！)")

In [ ]:
# 填空題 12.7.3：正確計算中間索引
# 任務：在給定的左右邊界下，使用小括號與整數除法計算 mid。
current_left = 6
current_right = 11

# 請填入正確的算式
mid_index = (___ + ___) // ___

print(f"區間 [{current_left}, {current_right}] 的中間索引為：", mid_index)

In [ ]:
# ==========================================
# [4] Code 練習題 12.7.3
# 任務說明：
# 給定多組 (left, right) 邊界對。
# 請使用列表生成式，計算出每一組對應的 mid 索引清單並印出。
#
# 【公開測試資料 1】
# bounds = [(0, 10), (3, 7), (5, 6), (8, 8)]
# 預期輸出：
# 各組中間索引： [5, 5, 5, 8]
#
# 【公開測試資料 2】
# bounds = [(1, 2), (2, 3)]
# 預期輸出：
# 各組中間索引： [1, 2]
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：
bounds = [(0, 10), (3, 7), (5, 6), (8, 8)]
mids = [(l + r) // 2 for l, r in bounds]
print("各組中間索引：", mids)

In [ ]:
# ==========================================
# [5] Code 挑戰題 12.7.3
# 任務說明：
# 請設計一個微型驗證程式：
# 給定任意長度為 N 的串列（例如 N = 9）。
# 印出最初的 mid 索引，並精確計算：以 mid 為界，
# 左半邊子串列（0 到 mid - 1）與右半邊子串列（mid + 1 到 N - 1）各自包含的元素個數。
# （本題為自由挑戰題，無公開測資，請依題目情境自行思考並撰寫完整程式碼）
# ==========================================
# 請在下方撰寫你的程式碼：
N = 9
left, right = 0, N - 1
mid = (left + right) // 2

left_count = mid - left
right_count = right - mid

print(f"總長度 {N}，正中央 mid = {mid}")
print(f"左半區間長度: {left_count}，右半區間長度: {right_count}")

### 12.7.4 區間縮小與指針跳躍鐵律：`left = mid + 1` 與 `right = mid - 1`，條件 `while left <= right:`

#### 1. 生活故事比喻：永遠不要回頭檢查已經排除的嫌疑犯
在猜數字遊戲中，如果秘密數字是 1 到 100，你猜了 50，莊家說「太大了」！
下一回合你會怎麼猜？
你下一回合的可疑範圍一定是「1 到 **49**」！你絕對不會傻傻把範圍設成「1 到 **50**」，因為 50 剛剛才被莊家當面宣判「不是答案」，如果你還把 50 留在搜尋範圍內，你就是在浪費生命！
在二分搜尋中也是一模一樣的鐵律：
- 當 `arr[mid] < target`：證明 `mid` 太小了，**`mid` 本身絕對不可能是答案！** 於是左邊界必須跨越它：**`left = mid + 1`**！
- 當 `arr[mid] > target`：證明 `mid` 太大了，**`mid` 本身也絕不可能是答案！** 於是右邊界必須越過它：**`right = mid - 1`**！

#### 2. 底層運作機制：破解無窮迴圈的生死線（`+1` 與 `-1`）
許多初學者手刻二分搜尋時，程式常常陷入「無窮迴圈死當」：
為什麼會死當？因為他們寫了：
```python
# 致命死結！
left = mid   # 或者 right = mid
```
當搜尋區間只剩下兩個相鄰元素時（例如 `left = 0, right = 1`）：
- 計算 `mid = (0 + 1) // 2 = 0`。
- 若條件判定需要往右走，你寫了 `left = mid`，結果 `left` 依然是 `0`！
- 下一輪：`left` 依然是 0，`right` 依然是 1，`mid` 依然是 0……
程式從此在 0 與 1 之間永遠鬼打牆旋轉，徹底卡死無窮迴圈！
只有加上 **`+ 1`** 與 **`- 1`**，才能保證每一輪區間長度**嚴格單調遞減至少 1**，絕不可能發生死結！

#### 3. 初學者常見陷阱：迴圈條件忘記寫等號 `while left < right:`
另一個極具殺傷力的邊界錯誤是：
```python
# 致命漏查！
while left < right:  # 漏掉了等號！
```
當搜尋區間收縮到只剩下「最後單一一個元素」時（例如 `left == right == 3`）：
此時唯一的目標候選人就在位置 3 上！如果你的迴圈條件是 `left < right`，迴圈在兩人相等時就提前終止退出了！最後這個元素連看都沒看就被當場遺漏，導致錯失正確答案！
因此標準二分搜尋的迴圈條件鐵律是：**`while left <= right:`**！

#### 4. APCS 實戰視野
`while left <= right:` 搭配 `left = mid + 1` 與 `right = mid - 1`，這三位一體的架構被資訊科學界公認為**「最完美的閉區間二分搜尋三劍客」**。背熟並理解這三處細節，考場上絕無死結與邊界遺漏之憂！

In [ ]:
# 範例 12.7.4：閉區間三劍客的收斂展示
# 示範在只有一個元素的極端邊界下，while left <= right: 的必要性
single_arr = [42]
target = 42

left = 0
right = len(single_arr) - 1  # left == 0, right == 0

found = False
# 必須有 <=，當 left == right == 0 時才能進入迴圈檢查！
while left <= right:
    mid = (left + right) // 2
    print(f"檢查中：left={left}, right={right}, mid={mid}, 數值={single_arr[mid]}")
    if single_arr[mid] == target:
        print("🎯 在最後單一元素時成功命中！")
        found = True
        break
    elif single_arr[mid] < target:
        left = mid + 1
    else:
        right = mid - 1

print("搜尋結束，命中狀態：", found)

In [ ]:
# 填空題 12.7.4：指針跳躍防死結填空
# 任務：填入正確的指針位移語句，確保區間單調縮小。
arr = [10, 20, 30, 40, 50]
target = 45

left = 0
right = len(arr) - 1

# 請填入包含等號的終止條件
while left ___ right:
    mid = (left + right) // 2
    if arr[mid] == target:
        print("找到目標！")
        break
    elif arr[mid] < target:
        # 太小，左界跳到 mid 右邊
        left = mid + ___
    else:
        # 太大，右界跳到 mid 左邊
        right = mid - ___

In [ ]:
# ==========================================
# [4] Code 練習題 12.7.4
# 任務說明：
# 給定已排序數列 primes = [2, 3, 5, 7, 11, 13, 17, 19]。
# 請使用標準的 while left <= right: 搭配 mid + 1 / mid - 1，
# 尋找 target 是否存在：
# 1. 若存在印出 "找到質數於索引：X"。
# 2. 若不存在印出 "不在質數清單中"。
#
# 【公開測試資料 1】
# target = 13
# 預期輸出：
# 找到質數於索引： 5
#
# 【公開測試資料 2】
# target = 9
# 預期輸出：
# 不在質數清單中
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：
primes = [2, 3, 5, 7, 11, 13, 17, 19]
target = 13

left = 0
right = len(primes) - 1
found_idx = -1

while left <= right:
    mid = (left + right) // 2
    if primes[mid] == target:
        found_idx = mid
        break
    elif primes[mid] < target:
        left = mid + 1
    else:
        right = mid - 1

if found_idx != -1:
    print("找到質數於索引：", found_idx)
else:
    print("不在質數清單中")

In [ ]:
# ==========================================
# [5] Code 挑戰題 12.7.4
# 任務說明：
# 請設計一個微型追蹤器，當搜尋不存在的目標時，印出退出 while 迴圈那一瞬間的 left 與 right 數值，
# 驗證退出時是否嚴格滿足 left == right + 1（盾牌交叉錯位）。
# 測試串列：data = [10, 20, 30, 40] ，搜尋 target = 25。
# （本題為自由挑戰題，無公開測資，請依題目情境自行思考並撰寫完整程式碼）
# ==========================================
# 請在下方撰寫你的程式碼：
data = [10, 20, 30, 40]
target = 25

l, r = 0, len(data) - 1
while l <= r:
    m = (l + r) // 2
    if data[m] == target:
        break
    elif data[m] < target:
        l = m + 1
    else:
        r = m - 1

print(f"搜尋完畢退出迴圈！最後狀態: left = {l}, right = {r}")
print(f"驗證是否嚴格滿足 left == right + 1：{l == r + 1}")

### 12.7.5 時間複雜度 $O(\log N)$ 的爆炸性威力：為什麼 10 億筆資料只需 30 次比對？

#### 1. 生活故事比喻：撕紙遊戲與全宇宙的原子總數
拿出一張普通的 A4 影印紙。如果你把它從中間撕成兩半，現在有 2 張；再撕成兩半，有 4 張；再撕一次，有 8 張。
如果讓你連續撕 30 次，這張紙會變成幾張？
答案是 $2^{30} \approx 1,073,741,824$（**整整十億七千萬張！**）。
現在把這個過程反過來：
如果你的面前堆著整整 **10 億張** 按編號排好的紙張，每一次二分搜尋，就把規模直接砍半！
第一刀剩下 5 億、第二刀剩下 2.5 億……
砍到第 30 刀時，原本廣袤無邊的 10 億筆資料，**瞬間只剩下最後 1 張紙！**
這就是以 2 為底的對數函數 **$O(\log_2 N)$** 震撼全球的指數級收斂速度！

#### 2. 底層運作機制：對數級時間複雜度推導
每一次比對後，問題規模從 $N$ 變為 $\frac{N}{2}$、$\frac{N}{4}$、$\dots$、$\frac{N}{2^k}$。
演算法終止的條件是問題規模縮小到 1：
$$\frac{N}{2^k} = 1 \implies 2^k = N \implies k = \log_2 N$$
因此，對於任意長度為 $N$ 的已排序串列，最差情況下的比對次數至多只有 $\lceil \log_2 N \rceil$ 次！

讓我們看一組令人熱血沸騰的數據對比：

| 資料量 $N$ | 線性搜尋最差次數 $O(N)$ | 二分搜尋最差次數 $O(\log_2 N)$ | 效能差距倍率 |
| :--- | :--- | :--- | :--- |
| **1,000** | 1,000 次 | **約 10 次** | 100 倍 |
| **1,000,000（百萬）** | 1,000,000 次 | **約 20 次** | 50,000 倍 |
| **1,000,000,000（十億）** | 10 億次（當機） | **約 30 次** | 3,300 萬倍！ |

#### 3. 初學者常見陷阱：高估了二分搜尋的次數
很多初學者在估算競賽時間時，以為「資料量好幾百萬筆，二分搜尋會不會跑很久？」
事實上，在電腦世界中，30 次運算只需要短短的 **0.0000003 秒**（不到一微秒）！二分搜尋在電腦眼中幾乎等同於「瞬間瞬移」。

#### 4. APCS 實戰視野
在 APCS 實作三級與四級題目中，當查詢次數高達 $Q = 10^5$ 次、資料庫長度 $N = 10^5$ 時：
- 若用線性搜尋：$Q \times N = 10^{10}$ 次運算，超時 100% 暴斃。
- 若用二分搜尋：$Q \times \log_2 N \approx 10^5 \times 17 \approx 1.7 \times 10^6$ 次運算，0.03 秒以滿分姿態極速通過！二分搜尋法就是化不可能為可能的競賽神技！

In [ ]:
# 範例 12.7.5：二分搜尋對數級威力實測
import math

# 觀察常見資料規模下的 log2 次數
scales = [100, 1000, 100000, 1000000, 1000000000]

print(f"{'資料規模 N':<15} | {'線性搜尋最差 (N)':<18} | {'二分搜尋最差 (log2 N)':<22}")
print("-" * 62)
for n in scales:
    log_steps = math.ceil(math.log2(n))
    print(f"{n:<17,d} | {n:<20,d} | {log_steps:<22d} 次")

# 實際在百萬筆虛擬序列中體驗瞬移（搜尋最末尾的元素）
# range 物件支援高效二分索引存取
huge_range = range(0, 2000000, 2)  # 一百萬個偶數
target = 1999998

l, r = 0, len(huge_range) - 1
count = 0

while l <= r:
    count += 1
    m = (l + r) // 2
    val = huge_range[m]
    if val == target:
        break
    elif val < target:
        l = m + 1
    else:
        r = m - 1

print(f"\n在 1,000,000 筆資料中尋找 {target}，僅花費 {count} 次比對即精確鎖定！")

In [ ]:
# 填空題 12.7.5：推算百萬數據的理論最大比對次數
# 任務：計算長度為 1,000,000 時，二分搜尋至多需要幾次折半。
import math

n = 1000000
# 請使用 math.ceil 與 math.log2 計算上限次數
max_comparisons = math.ceil(math.___(n))

print(f"一百萬筆已排序資料，二分搜尋至多只需 {max_comparisons} 次比對！")

In [ ]:
# ==========================================
# [4] Code 練習題 12.7.5
# 任務說明：
# 請撰寫程式：
# 給定任意正整數 N（代表資料量）。
# 輸出線性搜尋與二分搜尋在最差情況下的比對次數差額（線性次數 - 二分次數）。
#
# 【公開測試資料 1】
# N = 1024
# 預期輸出：
# 線性次數： 1024
# 二分次數： 10
# 節省比對次數： 1014
#
# 【公開測試資料 2】
# N = 64
# 預期輸出：
# 線性次數： 64
# 二分次數： 6
# 節省比對次數： 58
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：
import math
N = 1024
linear_cnt = N
bin_cnt = math.ceil(math.log2(N))
saved = linear_cnt - bin_cnt

print("線性次數：", linear_cnt)
print("二分次數：", bin_cnt)
print("節省比對次數：", saved)

In [ ]:
# ==========================================
# [5] Code 挑戰題 12.7.5
# 任務說明：
# 某伺服器每秒最多能承受 100,000 次元素比對。
# 假設資料量為 N = 10,000,000（一千萬）。
# 請計算：若使用二分搜尋，該伺服器在一秒內理論上「最多可處理幾次獨立使用者的查詢請求」？
# （本題為自由挑戰題，無公開測資，請依題目情境自行思考並撰寫完整程式碼）
# ==========================================
# 請在下方撰寫你的程式碼：
import math
N = 10000000
comparisons_per_search = math.ceil(math.log2(N))  # 每次約 24 次
server_capacity_per_sec = 100000
queries_per_sec = server_capacity_per_sec // comparisons_per_search

print(f"每次查詢需 {comparisons_per_search} 次比對。")
print(f"伺服器一秒內可輕鬆承受高達 {queries_per_sec} 次獨立查詢！")

### 12.7.6 手刻精確二分搜尋標準模板與找不到時回傳 -1

#### 1. 生活故事比喻：打造隨身攜帶的神級瑞士刀
在野外求生時，一位老練的冒險家不會每次生火時重新鑽木取火，他的腰間永遠掛著一把打磨得無比鋒利、隨抽隨用的瑞士多功能軍刀。
在資訊競賽與軟體面試中，**「手刻精確二分搜尋函數 `binary_search(arr, target)`」** 就是每一位頂尖選手腰間那把永不生鏽的瑞士軍刀！
無論題目包裝得多麼花俏，只要剝開外皮，核心往往都需要這 10 行最純粹、最穩健的二分骨架。

#### 2. 底層運作機制：世界級黃金標準封裝
讓我們將前面學到的所有心法，熔鑄為完美的標準函數：
```python
def binary_search(arr, target):
    left = 0
    right = len(arr) - 1
    
    while left <= right:
        mid = (left + right) // 2
        if arr[mid] == target:
            return mid        # 命中目標，回傳索引！
        elif arr[mid] < target:
            left = mid + 1    # 目標在右邊，左邊界推進
        else:
            right = mid - 1   # 目標在左邊，右邊界收縮
            
    return -1                 # 區間窮盡錯位，宣告查無此人
```
這段代碼具備三大無可撼動的優點：
1. **零外部依賴**：完全不依賴任何外部模組。
2. **零崩潰風險**：找不到時優雅回傳 `-1`，徹底杜絕 `ValueError`。
3. **極致時空效能**：空間複雜度 $O(1)$、時間複雜度 $O(\log N)$。

#### 3. 初學者常見陷阱：忘記傳回值是索引還是布林值
在撰寫函數時，必須想清楚題目要求的是：
- 要求回傳「在不在（`True / False`）」？
- 還是要求回傳「所在位置的索引（`index` 或 `-1`）」？
標準模板回傳的是索引。如果只需布林值，呼叫端只要寫 `binary_search(arr, target) != -1` 即可，通用性最高。

#### 4. APCS 實戰視野
熟練默寫這套 10 行標準模板，是征服 APCS 實作第三級的必經儀式。在考場緊張的高壓環境下，唯有將這套模板內化為肌肉記憶，你才能在幾十秒內迅速打出基石代碼，將寶貴的思考精力全數投注在更高階的邏輯設計上！

In [ ]:
# 範例 12.7.6：手刻精確二分搜尋標準模板實戰
def binary_search(arr, target):
    # 手刻標準精確二分搜尋
    # :param arr: 必須是已升序排序的串列
    # :param target: 搜尋目標
    # :return: 命中目標的索引，若不存在則回傳 -1
    left = 0
    right = len(arr) - 1
    
    while left <= right:
        mid = (left + right) // 2
        if arr[mid] == target:
            return mid
        elif arr[mid] < target:
            left = mid + 1
        else:
            right = mid - 1
            
    return -1

# 實測資料驗證
sorted_library = [102, 205, 318, 450, 560, 672, 789, 890, 999]
print("圖書館書籍編號清單：", sorted_library)

queries = [450, 999, 102, 500]

for q in queries:
    idx = binary_search(sorted_library, q)
    if idx != -1:
        print(f"  查詢 {q}: ✅ 成功找到！位於書架索引 {idx} (驗證: {sorted_library[idx]})")
    else:
        print(f"  查詢 {q}: ❌ 查無此書 (回傳 {idx})")

In [ ]:
# 填空題 12.7.6：完整默寫二分搜尋骨架
# 任務：補齊標準二分搜尋核心函數的關鍵語法。
def my_bsearch(arr, x):
    l = 0
    r = len(arr) - 1
    
    while l <= r:
        m = (l + r) // 2
        if arr[m] == x:
            return ___
        elif arr[m] < x:
            l = ___
        else:
            r = ___
    return ___

test_data = [2, 4, 6, 8, 10]
print("搜尋 8 位於索引：", my_bsearch(test_data, 8))
print("搜尋 5 回傳：", my_bsearch(test_data, 5))

In [ ]:
# ==========================================
# [4] Code 練習題 12.7.6
# 任務說明：
# 請利用標準 binary_search 模板，
# 解決會員登入驗證問題：
# 給定已排序會員名冊 member_ids 與登入者輸入的 ID login_id。
# 1. 若成功找到，印出 "登入成功，會員序號：X"（X 為索引）。
# 2. 若找不到，印出 "登入失敗，無效的帳號"。
#
# 【公開測試資料 1】
# member_ids = [101, 204, 305, 408, 512]
# login_id = 305
# 預期輸出：
# 登入成功，會員序號： 2
#
# 【公開測試資料 2】
# member_ids = [101, 204, 305, 408, 512]
# login_id = 999
# 預期輸出：
# 登入失敗，無效的帳號
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：
def binary_search(arr, target):
    l, r = 0, len(arr) - 1
    while l <= r:
        m = (l + r) // 2
        if arr[m] == target:
            return m
        elif arr[m] < target:
            l = m + 1
        else:
            r = m - 1
    return -1

member_ids = [101, 204, 305, 408, 512]
login_id = 305

idx = binary_search(member_ids, login_id)
if idx != -1:
    print("登入成功，會員序號：", idx)
else:
    print("登入失敗，無效的帳號")

In [ ]:
# ==========================================
# [5] Code 挑戰題 12.7.6
# 任務說明：
# 某銀行帳戶系統有一串已排序的帳號黑名單：
# blacklist = [10001, 10050, 10200, 10550, 10900, 11000]
# 某日交易紀錄中出現了一批轉帳帳號：transfers = [10050, 10300, 10900, 12000]
# 請撰寫程式，利用手刻 binary_search 逐一檢驗轉帳帳號，
# 收集所有「出現在黑名單中」的危險帳號並印出警報。
# （本題為自由挑戰題，無公開測資，請依題目情境自行思考並撰寫完整程式碼）
# ==========================================
# 請在下方撰寫你的程式碼：
def binary_search(arr, target):
    l, r = 0, len(arr) - 1
    while l <= r:
        m = (l + r) // 2
        if arr[m] == target:
            return m
        elif arr[m] < target:
            l = m + 1
        else:
            r = m - 1
    return -1

blacklist = [10001, 10050, 10200, 10550, 10900, 11000]
transfers = [10050, 10300, 10900, 12000]
intercepted = []

for acc in transfers:
    if binary_search(blacklist, acc) != -1:
        intercepted.append(acc)

print("🚨 攔截到黑名單危險轉帳帳號：", intercepted)

### 學習總結與通關回顧

恭喜你順利通關 **12.7 二分搜尋法手刻演算法一：猜數字模型與精確匹配**！

在本單元中，你完成了演算法學習史上最偉大的一次認知飛躍：
- **二分搜尋的第一鐵律**：
  - **資料必須已經嚴格排序！** 唯有具備單調性，折半淘汰才具備數學必然性。
- **猜數字物理模型**：
  - 左右雙指標 `left, right` 鎖定可疑閉區間 `[left, right]`。
  - 中間點計算：`mid = (left + right) // 2`，小括號防禦優先級。
- **閉區間三劍客（防死結神技）**：
  - 迴圈條件：`while left <= right:`，等號守護最後單一元素。
  - 邊界跳躍：`left = mid + 1` 與 `right = mid - 1`，保證區間長度單調遞減。
- **$O(\log N)$ 的對數級神速**：
  - 每回合排除一半候選者，10 億筆資料僅需 30 次比對即可精準破案。
- **標準模板手刻實踐**：
  - 命中回傳索引，未果安全回傳 `-1`，兼具極致效能與工程安全性。

---
**下一關預告**：如果數列中有「重複出現的元素」（例如串列裡有 5 個連續的 40 分），精確二分搜尋會回傳哪一個？如果我們想找「第一個大於等於目標」的位置該怎麼辦？下一節 **12.8 二分搜尋法手刻演算法二：邊界二分搜尋（Lower / Upper Bound）** 將帶你征服最精密的二分進階境界！